Bank Marketing Data Cleaning   ---

# STEP 1:loading data

In [2]:
import pandas as pd 

In [4]:
# ── STEP 1: LOAD THE DATA ─────────────────────────────────────
df = pd.read_csv('bank-additional-full.csv', sep=';')

print("Rows:", len(df))
print("Columns:", len(df.columns))
print()

Rows: 41188
Columns: 21



# ── STEP 2: EXPLORE THE DATA ───

In [8]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [9]:
# Check how many Yes/No in target column
print("Subscribed (y) counts:")
print(df['y'].value_counts())
print()

Subscribed (y) counts:
y
no     36548
yes     4640
Name: count, dtype: int64



# ── STEP 3: CHECK FOR PROBLEMS

In [10]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

Missing values per column:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64



In [11]:
# This dataset uses 'unknown' instead of blank — let's count them
print("'Unknown' values per column:")
for col in df.columns:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        print(f"  {col}: {count}")
print()

'Unknown' values per column:
  job: 330
  marital: 80
  education: 1731
  default: 8597
  housing: 990
  loan: 990



# ── STEP 4: CLEAN THE DATA ─

In [12]:
#Replace 'unknown' text with NaN (proper missing value)
df.replace('unknown', pd.NA, inplace=True)
print("✅ Replaced 'unknown' with NaN")

✅ Replaced 'unknown' with NaN


In [13]:
#  Drop rows where job, marital, or education is missing
df.dropna(subset=['job', 'marital', 'education'], inplace=True)
print("✅ Removed rows with missing job / marital / education")

✅ Removed rows with missing job / marital / education


In [14]:
# Fill the remaining missing values with the most common value
df['default'] = df['default'].fillna(df['default'].mode()[0])
df['housing'] = df['housing'].fillna(df['housing'].mode()[0])
df['loan'] = df['loan'].fillna(df['loan'].mode()[0])
print("✅ Filled missing default / housing / loan values")

✅ Filled missing default / housing / loan values


In [15]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)
print("✅ Removed duplicate rows")

✅ Removed duplicate rows


In [16]:
#  Rename the target column 'y' to something clearer
df.rename(columns={'y': 'subscribed'}, inplace=True)
print("✅ Renamed 'y' column to 'subscribed'")

✅ Renamed 'y' column to 'subscribed'


In [17]:
#  Add a numeric version of subscribed (1 = yes, 0 = no)
#     Needed for SQL AVG() and Power BI calculations
df['subscribed_flag'] = df['subscribed'].map({'yes': 1, 'no': 0})
print("✅ Added 'subscribed_flag' column (1=yes, 0=no)")

✅ Added 'subscribed_flag' column (1=yes, 0=no)


In [18]:
# Fix column names — replace dots with underscores
#     (SQL doesn't like dots in column names)
df.columns = df.columns.str.replace('.', '_', regex=False)
print("✅ Fixed column names (dots replaced with underscores)")
print()

✅ Fixed column names (dots replaced with underscores)



# ── STEP 5: ADD USEFUL COLUMNS 

In [19]:

# Age group — for segmentation in Power BI
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=['Under 25', '25-35', '35-50', '50-65', '65+']
).astype(str)
print("✅ Added age_group")

✅ Added age_group


In [21]:
# Call bucket — group number of calls made
df['call_bucket'] = pd.cut(
    df['campaign'],
    bins=[0, 1, 3, 5, 100],
    labels=['1 call', '2-3 calls', '4-5 calls', '6+ calls']
).astype(str)
print("✅ Added call_bucket")


✅ Added call_bucket


In [22]:
# Month number — so charts sort Jan→Dec correctly
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}
df['month_num'] = df['month'].map(month_map)
print("✅ Added month_num")
print()


✅ Added month_num



# ── STEP 6: FINAL CHECK 

In [26]:

print("Clean dataset shape:", df.shape)
print()
print("Subscribed counts after cleaning:")
print(df['subscribed'].value_counts())
print()

Clean dataset shape: (39178, 25)

Subscribed counts after cleaning:
subscribed
no     34819
yes     4359
Name: count, dtype: int64



# ── STEP 7: SAVE THE FILE

In [25]:

df.to_csv('bank_cleaned.csv', index=False)

print("✅ Saved as bank_cleaned.csv")
print()

✅ Saved as bank_cleaned.csv

